# CEX Raw Data Lake Builder — Binance Futures + Coinbase (1h)

This notebook is the entry point for building the **raw data lake** used in the empirical section. It downloads and standardizes hourly market data from two sources, applies basic QA checks, and writes provenance artifacts (manifests, environment fingerprint, file hashes) so each run is auditable.

## Scope (datasets produced)
**Binance Futures**
- **Klines (OHLCV) — 1h**
- **Funding rate**
- **Open interest snapshot**

**Coinbase Exchange**
- **Candles — 1h** (used as a benchmark series)
- QA on the expected hourly index
- Targeted re-fetch of missing windows (“holes”)

## Time conventions
- All timestamps are handled in **UTC**.
- All extraction windows follow the half-open interval convention **[START, END_EXCL)**:
  values satisfy `ts >= START` and `ts < END_EXCL`.
  This prevents overlap when concatenating adjacent runs.

## Technical approach (high-level)
To keep long downloads reliable and publication-ready, the notebook implements:
- **Robust HTTP**: persistent sessions, connection pooling, retries, exponential backoff, jitter, and support for `Retry-After` when rate-limited.
- **Throttling**: a minimum interval between requests to reduce the chance of hitting provider limits.
- **Chunking / paging**: requests are split over time to avoid timeouts and large payloads.
- **Traceability**: manifests include run parameters, an environment fingerprint, and **SHA256 hashes** of the produced files.

## What the notebook does (in order)
1) **Setup**: reads configuration, defines `[START, END_EXCL)`, and prepares the `data/` folder structure.
2) **Download (Binance Futures)**:
   - fetches 1h klines over the window,
   - fetches funding observations and stores both the raw timestamps and an hourly bucket,
   - fetches open interest snapshots.
3) **Download (Coinbase Exchange)**:
   - fetches 1h candles for the same window (benchmark),
   - de-duplicates and checks the expected hourly index,
   - automatically re-fetches the missing windows detected by QA.
4) **QA and exports**: writes raw extracts plus QA summaries.
5) **Provenance**: writes manifests and computes SHA256 hashes for all artifacts.

## Outputs (under `data/`)
This notebook produces (directly or via its dataset routines):
- `data/raw/…` — raw responses / raw extracts (audit trail)
- `data/processed/…` — standardized 1h tables (analysis-ready)
- `data/qa/…` — QA summaries (missing hours, duplicates, bounds)
- `data/manifests/…` — run manifests (parameters, environment fingerprint, SHA256)

## Dataset-specific notes
- **Funding (Binance)**: the notebook preserves the raw event timestamp and also assigns an hourly bucket (`ts` on the hourly grid). Duplicates are removed after bucketing so merges remain stable.
- **Coinbase candles**: the notebook enforces an expected hourly index and can re-fetch missing windows to improve completeness.

## Reproducibility note
The pipeline is reproducible as a procedure (same code + parameters). Exact bit-for-bit reproducibility depends on providers returning identical historical data for the same window. Manifests and hashes record the exact artifacts generated for each run.

## Quick notes
- Prefer running the notebook top-to-bottom to keep the data lake consistent across sources.
- If something looks off, check the QA outputs first (missing hours, duplicates, time bounds), then confirm the window and symbols.
- Keep `END_EXCL` exclusive so adjacent windows can be concatenated without overlap.

In [ ]:
# ============================================================
# CEX Raw Data Lake Builder — DEFINITIF (publication-ready)
# Binance Futures: klines (1h), fundingRate, openInterest snapshot
# Coinbase Exchange: candles (benchmark, 1h) + QA + refetch ciblé des trous
#
# ✅ Sortie écrivable auto (Jupyter/Anaconda)
# ✅ HTTP robuste: Session + pool + retries + backoff + jitter + Retry-After
# ✅ Throttle (min interval)
# ✅ Chunking/paging (évite timeouts)
# ✅ Manifests + empreinte d'environnement + SHA256 des fichiers
# ✅ Convention TEMPS: UTC + borne droite EXCLUSIVE partout (>= start, < end)
# ✅ Funding: conserve timestamp brut + bucket horaire (round('h')) + dedup
# ✅ Coinbase: dedup + QA index horaire attendu + refetch automatique fenêtres manquantes
# ============================================================

import os, json, time, sys, platform, hashlib, random
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime, timezone

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import pandas as pd
import numpy as np


# ----------------------------
# CONFIG
# ----------------------------
@dataclass
class Config:
    # assets
    symbol_binance: str = "ETHUSDT"
    product_coinbase: str = "ETH-USD"

    # time
    interval: str = "1h"
    start: str = "2021-01-01T00:00:00Z"
    end:   str = "2025-12-01T00:00:00Z"   # borne droite EXCLUSIVE (publication-grade)

    # output
    out_root: str | None = None  # None => auto-pick writable folder

    # endpoints
    binance_base: str = "https://fapi.binance.com"
    coinbase_base: str = "https://api.exchange.coinbase.com"

    # api politeness / robustness
    min_request_interval_s: float = 0.20
    timeout_connect_s: float = 10
    timeout_read_s: float = 120

    # retries (network + 429 + 5xx)
    retries_total: int = 8
    backoff_factor: float = 1.0

    # paging/chunking
    binance_klines_limit: int = 1000
    binance_funding_limit: int = 1000
    funding_chunk_days: int = 330  # ~1000 funding events (8h) ≈ 333 days

    # coinbase (max 300 points / req)
    coinbase_granularity_s: int = 3600
    coinbase_max_points: int = 300
    coinbase_refetch_passes: int = 2       # combien de passes de réparation
    coinbase_refetch_pad_hours: int = 2    # padding autour des trous (gère inclusivité)


CFG = Config()


# ----------------------------
# UTILS
# ----------------------------
def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

def pick_out_root(preferred="/content/raw_data_lake", fallback_name="raw_data_lake") -> str:
    candidates = [Path(preferred), Path.cwd() / fallback_name, Path.home() / fallback_name]
    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            test = p / ".write_test"
            test.write_text("ok", encoding="utf-8")
            test.unlink()
            return str(p)
        except Exception:
            continue
    raise OSError("Aucun dossier écrivable trouvé. Spécifie CFG.out_root explicitement.")

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def to_ts_utc(iso_z: str) -> pd.Timestamp:
    return pd.Timestamp(iso_z).tz_convert("UTC")

def utc_ms(iso_z: str) -> int:
    return int(to_ts_utc(iso_z).value // 10**6)

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def env_fingerprint() -> dict:
    return {
        "collected_at_utc": now_iso(),
        "python": sys.version.replace("\n", " "),
        "platform": platform.platform(),
        "pandas": getattr(pd, "__version__", None),
        "numpy": getattr(np, "__version__", None),
        "requests": getattr(requests, "__version__", None),
    }

def write_json(path: str, payload: dict):
    ensure_dir(str(Path(path).parent))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

def save_df(df: pd.DataFrame, folder: str, name: str) -> dict:
    ensure_dir(folder)
    out = {}

    csv_path = os.path.join(folder, f"{name}.csv")
    df.to_csv(csv_path, index=False)
    out["csv_path"] = csv_path
    out["csv_sha256"] = sha256_file(csv_path)

    parquet_path = os.path.join(folder, f"{name}.parquet")
    try:
        df.to_parquet(parquet_path, index=False)
        out["parquet_path"] = parquet_path
        out["parquet_sha256"] = sha256_file(parquet_path)
    except Exception as e:
        out["parquet_path"] = None
        out["parquet_error"] = repr(e)

    return out

def filter_end_exclusive(df: pd.DataFrame, col: str, start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> pd.DataFrame:
    return df[(df[col] >= start_ts) & (df[col] < end_ts)].copy()

def expected_counts_1h(start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> dict:
    hours = int((end_ts - start_ts) / pd.Timedelta(hours=1))
    fund = hours // 8
    return {"expected_1h_candles": hours, "expected_funding_8h": fund}


# ----------------------------
# RATE LIMITER + HTTP CLIENT
# ----------------------------
class RateLimiter:
    def __init__(self, min_interval_s: float):
        self.min_interval_s = float(min_interval_s)
        self._last = 0.0

    def wait(self):
        now = time.time()
        delta = now - self._last
        if delta < self.min_interval_s:
            time.sleep(self.min_interval_s - delta)
        self._last = time.time()

class HttpClient:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.rate = RateLimiter(cfg.min_request_interval_s)
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "raw-data-lake-builder/definitive-publication-ready"})

        retry = Retry(
            total=cfg.retries_total,
            connect=cfg.retries_total,
            read=cfg.retries_total,
            backoff_factor=cfg.backoff_factor,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=("GET",),
            raise_on_status=False,
            respect_retry_after_header=True,
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

        self.requests_made = 0
        self.errors_transient = 0

    def get_json(self, url: str, params: dict, timeout=None):
        if timeout is None:
            timeout = (self.cfg.timeout_connect_s, self.cfg.timeout_read_s)
        self.rate.wait()
        self.requests_made += 1
        r = self.session.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        return r.json(), {
            "url": url,
            "params": params,
            "status": r.status_code,
            "server_date": r.headers.get("Date"),
        }

    def safe_get_json(self, url: str, params: dict, tag: str, timeout=None, tries: int = 6):
        last = None
        for i in range(tries):
            try:
                return self.get_json(url, params=params, timeout=timeout)
            except (requests.exceptions.ReadTimeout,
                    requests.exceptions.ConnectTimeout,
                    requests.exceptions.SSLError,
                    requests.exceptions.ConnectionError) as e:
                self.errors_transient += 1
                last = e
                wait = (2 ** i) * 1.0 + random.random() * 0.5
                print(f"[{tag}] transient error -> retry in {wait:.2f}s | {repr(e)}")
                time.sleep(wait)
        raise last


# ----------------------------
# INTERVAL MS (for klines cursor step)
# ----------------------------
INTERVAL_MS = {
    "1m": 60_000, "3m": 180_000, "5m": 300_000, "15m": 900_000, "30m": 1_800_000,
    "1h": 3_600_000, "2h": 7_200_000, "4h": 14_400_000, "6h": 21_600_000,
    "8h": 28_800_000, "12h": 43_200_000, "1d": 86_400_000
}


# ============================================================
# BINANCE: KLINES
# ============================================================
def download_binance_klines(client: HttpClient, cfg: Config) -> pd.DataFrame:
    start_ts = to_ts_utc(cfg.start)
    end_ts = to_ts_utc(cfg.end)
    start_ms = utc_ms(cfg.start)
    end_ms = utc_ms(cfg.end)

    out_dir = os.path.join(cfg.out_root, "raw", "cex", "binance_futures", f"klines_{cfg.interval}", cfg.symbol_binance)
    ensure_dir(out_dir)
    pages_path = os.path.join(out_dir, "pages.jsonl")

    url = f"{cfg.binance_base}/fapi/v1/klines"
    step = INTERVAL_MS.get(cfg.interval, 3_600_000)

    rows = []
    cursor = start_ms
    page = 0

    while cursor < end_ms:
        params = {
            "symbol": cfg.symbol_binance,
            "interval": cfg.interval,
            "limit": int(cfg.binance_klines_limit),
            "startTime": int(cursor),
            "endTime": int(end_ms),
        }
        data, meta = client.safe_get_json(url, params=params, tag="klines")

        with open(pages_path, "a", encoding="utf-8") as f:
            f.write(json.dumps({"page": page, "meta": meta, "n": len(data)}, ensure_ascii=False) + "\n")

        if not data:
            break

        rows.extend(data)
        last_open = int(data[-1][0])
        cursor = last_open + step
        page += 1

        if page % 25 == 0:
            print(f"[klines] pages={page} rows={len(rows)} cursor={pd.to_datetime(cursor, unit='ms', utc=True)}")

    cols = [
        "open_time","open","high","low","close","volume","close_time","quote_asset_volume","num_trades",
        "taker_buy_base","taker_buy_quote","ignore"
    ]
    df = pd.DataFrame(rows, columns=cols)

    if not df.empty:
        df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
        df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)
        for c in ["open","high","low","close","volume","quote_asset_volume","taker_buy_base","taker_buy_quote"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["num_trades"] = pd.to_numeric(df["num_trades"], errors="coerce").astype("Int64")

        # Dedup safety + bornage publication-grade (end exclusive)
        df = df.sort_values("open_time").drop_duplicates(subset=["open_time"], keep="last").reset_index(drop=True)
        df = filter_end_exclusive(df, "open_time", start_ts, end_ts)

    files = save_df(df, out_dir, "klines_raw")

    manifest = {
        "dataset": "binance_futures_klines",
        "endpoint": "/fapi/v1/klines",
        "symbol": cfg.symbol_binance,
        "interval": cfg.interval,
        "start": cfg.start,
        "end_exclusive": cfg.end,
        "rows": int(df.shape[0]),
        "files": files,
        "pages_jsonl": pages_path,
        "environment": env_fingerprint(),
        "http_summary": {"requests_made": client.requests_made, "transient_errors": client.errors_transient},
        "notes": "Filtered to [start, end) to avoid off-by-one at the boundary.",
    }
    write_json(os.path.join(out_dir, "manifest.json"), manifest)
    return df


# ============================================================
# BINANCE: FUNDING RATE (chunked) + bucket hour (round) for alignment
# ============================================================
def download_binance_funding(client: HttpClient, cfg: Config) -> pd.DataFrame:
    start_ts = to_ts_utc(cfg.start)
    end_ts = to_ts_utc(cfg.end)
    start_ms = utc_ms(cfg.start)
    end_ms = utc_ms(cfg.end)

    out_dir = os.path.join(cfg.out_root, "raw", "cex", "binance_futures", "funding", cfg.symbol_binance)
    ensure_dir(out_dir)
    pages_path = os.path.join(out_dir, "pages.jsonl")

    url = f"{cfg.binance_base}/fapi/v1/fundingRate"
    chunk_ms = int(cfg.funding_chunk_days * 24 * 3600_000)

    rows = []
    cursor = start_ms
    page = 0

    while cursor < end_ms:
        chunk_end = min(end_ms, cursor + chunk_ms)
        params = {
            "symbol": cfg.symbol_binance,
            "limit": int(cfg.binance_funding_limit),
            "startTime": int(cursor),
            "endTime": int(chunk_end),
        }
        data, meta = client.safe_get_json(url, params=params, tag="funding")

        with open(pages_path, "a", encoding="utf-8") as f:
            f.write(json.dumps({"page": page, "meta": meta, "n": len(data)}, ensure_ascii=False) + "\n")

        if not data:
            cursor = int(chunk_end) + 1
            page += 1
            continue

        rows.extend(data)
        last_ts = int(data[-1]["fundingTime"])
        cursor = last_ts + 1
        page += 1

        if page % 25 == 0:
            print(f"[funding] pages={page} rows={len(rows)} cursor={pd.to_datetime(cursor, unit='ms', utc=True)}")

    df = pd.DataFrame(rows)

    if not df.empty:
        # Keep raw timestamp (ms jitter)
        df["fundingTime_raw"] = pd.to_datetime(df["fundingTime"], unit="ms", utc=True)
        df["fundingRate"] = pd.to_numeric(df["fundingRate"], errors="coerce")
        if "markPrice" in df.columns:
            df["markPrice"] = pd.to_numeric(df["markPrice"], errors="coerce")

        # Publication-grade: bucket to the nearest hour to avoid ms jitter issues on joins
        df["fundingTime"] = df["fundingTime_raw"].dt.round("h")

        # Dedup on bucketed hour + bornage [start, end)
        df = df.sort_values("fundingTime").drop_duplicates(subset=["fundingTime"], keep="last").reset_index(drop=True)
        df = filter_end_exclusive(df, "fundingTime", start_ts, end_ts)

        # Keep a clean, explicit schema
        keep_cols = ["fundingTime", "fundingTime_raw", "fundingRate"]
        if "markPrice" in df.columns:
            keep_cols.append("markPrice")
        if "symbol" in df.columns:
            keep_cols.append("symbol")
        df = df[keep_cols].copy()

    files = save_df(df, out_dir, "funding_raw")

    manifest = {
        "dataset": "binance_futures_fundingRate",
        "endpoint": "/fapi/v1/fundingRate",
        "symbol": cfg.symbol_binance,
        "start": cfg.start,
        "end_exclusive": cfg.end,
        "rows": int(df.shape[0]),
        "chunk_days": cfg.funding_chunk_days,
        "files": files,
        "pages_jsonl": pages_path,
        "environment": env_fingerprint(),
        "http_summary": {"requests_made": client.requests_made, "transient_errors": client.errors_transient},
        "notes": "fundingTime_raw preserved; fundingTime bucketed to hour using round('h') for stable hourly alignment.",
    }
    write_json(os.path.join(out_dir, "manifest.json"), manifest)
    return df


# ============================================================
# BINANCE: OPEN INTEREST SNAPSHOT + 30 days historic
# ============================================================
def snapshot_binance_open_interest(client: HttpClient, cfg: Config) -> dict:
    out_dir = os.path.join(cfg.out_root, "raw", "cex", "binance_futures", "open_interest", cfg.symbol_binance)
    ensure_dir(out_dir)

    url = f"{cfg.binance_base}/fapi/v1/openInterest"
    data, meta = client.safe_get_json(url, params={"symbol": cfg.symbol_binance}, tag="open_interest")

    ts = int(time.time())
    path = os.path.join(out_dir, f"open_interest_snapshot_{ts}.json")
    write_json(path, {"meta": meta, "data": data})

    manifest = {
        "dataset": "binance_futures_openInterest_snapshot",
        "endpoint": "/fapi/v1/openInterest",
        "symbol": cfg.symbol_binance,
        "snapshot_unix": ts,
        "file": {"json_path": path, "json_sha256": sha256_file(path)},
        "environment": env_fingerprint(),
        "http_summary": {"requests_made": client.requests_made, "transient_errors": client.errors_transient},
        "notes": "Snapshot only (not historical).",
    }
    write_json(os.path.join(out_dir, "manifest.json"), manifest)
    return data

# ============================================================
# COINBASE: CANDLES (chunked) + QA + refetch trous
# ============================================================
def download_coinbase_candles(client: HttpClient, cfg: Config) -> pd.DataFrame:
    start_ts = to_ts_utc(cfg.start)
    end_ts = to_ts_utc(cfg.end)

    out_dir = os.path.join(cfg.out_root, "raw", "benchmarks", "coinbase_exchange",
                           f"candles_{cfg.coinbase_granularity_s}", cfg.product_coinbase)
    ensure_dir(out_dir)
    pages_path = os.path.join(out_dir, "pages.jsonl")

    url = f"{cfg.coinbase_base}/products/{cfg.product_coinbase}/candles"
    gran = int(cfg.coinbase_granularity_s)
    chunk_seconds = gran * int(cfg.coinbase_max_points)

    cb_timeout = (cfg.timeout_connect_s, max(cfg.timeout_read_s, 180))

    cur = start_ts          # ✅ keep tz-aware Timestamp
    end = end_ts            # ✅ keep tz-aware Timestamp

    rows = []
    page = 0

    while cur < end:
        nxt = min(end, cur + pd.Timedelta(seconds=chunk_seconds))

        params = {
            "start": cur.isoformat().replace("+00:00", "Z"),
            "end":   nxt.isoformat().replace("+00:00", "Z"),
            "granularity": gran,
        }

        data, meta = client.safe_get_json(url, params=params, tag="coinbase", timeout=cb_timeout)

        with open(pages_path, "a", encoding="utf-8") as f:
            f.write(json.dumps({"page": page, "meta": meta, "n": len(data)}, ensure_ascii=False) + "\n")

        rows.extend(data)
        cur = nxt
        page += 1

        if page % 50 == 0:
            print(f"[coinbase] pages={page} rows={len(rows)} cur={cur}")

    df = pd.DataFrame(rows, columns=["time","low","high","open","close","volume"])
    if not df.empty:
        df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
        for c in ["low","high","open","close","volume"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df = df.sort_values("time").drop_duplicates(subset=["time"], keep="last").reset_index(drop=True)
        df = filter_end_exclusive(df, "time", start_ts, end_ts)

    files = save_df(df, out_dir, "candles_raw")

    manifest = {
        "dataset": "coinbase_exchange_candles",
        "endpoint": f"/products/{cfg.product_coinbase}/candles",
        "product_id": cfg.product_coinbase,
        "granularity_seconds": gran,
        "start": cfg.start,
        "end_exclusive": cfg.end,
        "rows": int(df.shape[0]),
        "files": files,
        "pages_jsonl": pages_path,
        "environment": env_fingerprint(),
        "http_summary": {"requests_made": client.requests_made, "transient_errors": client.errors_transient},
        "notes": "Downloaded in <=300-point chunks; dedup by time; filtered to [start, end).",
    }
    write_json(os.path.join(out_dir, "manifest.json"), manifest)
    return df

def hourly_index_expected(cfg: Config) -> pd.DatetimeIndex:
    start_ts = to_ts_utc(cfg.start)
    end_ts = to_ts_utc(cfg.end)
    # end exclusive => last point = end - 1h
    return pd.date_range(start=start_ts, end=end_ts - pd.Timedelta(hours=1), freq="h", tz="UTC")

def list_missing_hours(df: pd.DataFrame, time_col: str, idx: pd.DatetimeIndex) -> list[pd.Timestamp]:
    present = pd.DatetimeIndex(df[time_col].dropna().unique()).tz_convert("UTC")
    missing = idx.difference(present)
    return list(missing)

def group_consecutive_hours(hours: list[pd.Timestamp]) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    if not hours:
        return []
    hours_sorted = sorted(pd.DatetimeIndex(hours))
    groups = []
    start = hours_sorted[0]
    prev = hours_sorted[0]
    for t in hours_sorted[1:]:
        if t == prev + pd.Timedelta(hours=1):
            prev = t
        else:
            groups.append((start, prev))
            start = t
            prev = t
    groups.append((start, prev))
    return groups

def refetch_coinbase_windows(client: HttpClient, cfg: Config, df: pd.DataFrame, windows: list[tuple[pd.Timestamp, pd.Timestamp]]) -> pd.DataFrame:
    if not windows:
        return df

    url = f"{cfg.coinbase_base}/products/{cfg.product_coinbase}/candles"
    gran = int(cfg.coinbase_granularity_s)
    cb_timeout = (cfg.timeout_connect_s, max(cfg.timeout_read_s, 180))
    pad = int(cfg.coinbase_refetch_pad_hours)

    new_rows = []
    for (a, b) in windows:
        start = a - pd.Timedelta(hours=pad)
        end = b + pd.Timedelta(hours=pad + 1)  # +1h to cover b candle safely
        params = {
            "start": start.isoformat().replace("+00:00", "Z"),
            "end": end.isoformat().replace("+00:00", "Z"),
            "granularity": gran,
        }
        data, meta = client.safe_get_json(url, params=params, tag="coinbase_refetch", timeout=cb_timeout)
        new_rows.extend(data)

    patch = pd.DataFrame(new_rows, columns=["time","low","high","open","close","volume"])
    if patch.empty:
        return df

    patch["time"] = pd.to_datetime(patch["time"], unit="s", utc=True)
    for c in ["low","high","open","close","volume"]:
        patch[c] = pd.to_numeric(patch[c], errors="coerce")

    start_ts = to_ts_utc(cfg.start)
    end_ts = to_ts_utc(cfg.end)

    # Merge + dedup + bornage [start, end)
    merged = pd.concat([df, patch], ignore_index=True)
    merged = merged.sort_values("time").drop_duplicates(subset=["time"], keep="last").reset_index(drop=True)
    merged = filter_end_exclusive(merged, "time", start_ts, end_ts)

    return merged

def repair_coinbase_if_needed(client: HttpClient, cfg: Config, df_cb: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    idx = hourly_index_expected(cfg)
    missing = list_missing_hours(df_cb, "time", idx)

    audit = {
        "initial_rows": int(len(df_cb)),
        "expected_rows": int(len(idx)),
        "initial_missing_hours": [str(x) for x in missing[:50]],
        "initial_missing_count": int(len(missing)),
        "refetch_passes": [],
    }

    cur = df_cb
    for p in range(int(cfg.coinbase_refetch_passes)):
        if not missing:
            break
        windows = group_consecutive_hours(missing)
        cur = refetch_coinbase_windows(client, cfg, cur, windows)
        missing = list_missing_hours(cur, "time", idx)

        audit["refetch_passes"].append({
            "pass": p + 1,
            "windows": [(str(a), str(b)) for (a, b) in windows],
            "rows_after": int(len(cur)),
            "missing_after_count": int(len(missing)),
            "missing_after_first_50": [str(x) for x in missing[:50]],
        })

    audit["final_rows"] = int(len(cur))
    audit["final_missing_count"] = int(len(missing))
    audit["final_missing_hours_first_200"] = [str(x) for x in missing[:200]]
    return cur, audit


# ============================================================
# NORMALIZATION + QA
# ============================================================
def normalize_binance_core(cfg: Config, klines: pd.DataFrame, funding: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    start_ts = to_ts_utc(cfg.start)
    end_ts = to_ts_utc(cfg.end)
    idx = hourly_index_expected(cfg)

    # core from klines
    df = klines.copy()
    df = df.rename(columns={"open_time": "date"})
    df = df[["date","open","high","low","close","volume","quote_asset_volume","num_trades","taker_buy_base","taker_buy_quote"]]
    df = df.sort_values("date").reset_index(drop=True)

    # enforce [start, end) (double safety)
    df = filter_end_exclusive(df, "date", start_ts, end_ts)

    # reindex to expected hourly grid
    df = df.set_index("date").reindex(idx).rename_axis("date").reset_index()

    df["volume_usd"] = df["quote_asset_volume"].astype(float)
    df["log_returns"] = np.log(df["close"] / df["close"].shift(1))

    # funding alignment (bucketed on fundingTime already)
    if funding is None or funding.empty:
        df["funding_rate"] = np.nan
    else:
        s = funding.set_index("fundingTime")["fundingRate"].sort_index()
        s = s.reindex(idx, method="ffill")
        df["funding_rate"] = s.values

    # QA flags
    df["flag_close_missing"] = df["close"].isna()
    df["flag_volume_zero"] = (df["volume_usd"] == 0)

    # QA report
    exp = expected_counts_1h(start_ts, end_ts)
    qa = {
        "start": cfg.start,
        "end_exclusive": cfg.end,
        **exp,
        "actual_hours_index": int(len(idx)),
        "missing_close_hours": int(df["close"].isna().sum()),
        "missing_close_first_20": [str(x) for x in df.loc[df["close"].isna(), "date"].head(20).to_list()],
        "funding_rate_missing_hours": int(pd.Series(df["funding_rate"]).isna().sum()),
        "funding_rate_missing_first_20": [str(x) for x in df.loc[pd.Series(df["funding_rate"]).isna(), "date"].head(20).to_list()],
    }
    return df, qa


# ============================================================
# RUN
# ============================================================
def run(cfg: Config):
    if not cfg.out_root:
        cfg.out_root = pick_out_root()
    print("✅ out_root =", cfg.out_root)

    start_ts = to_ts_utc(cfg.start)
    end_ts = to_ts_utc(cfg.end)
    print("📌 Convention temps: UTC + [start, end) avec end EXCLUSIVE")
    print("   start =", start_ts, "| end =", end_ts)

    client = HttpClient(cfg)

    # Connectivity checks
    client.safe_get_json(f"{cfg.binance_base}/fapi/v1/ping", params={}, tag="binance_ping")
    print("✅ Binance ping OK")

    # 1) Binance klines
    print("\nDownloading Binance klines...")
    kl = download_binance_klines(client, cfg)
    print("✅ klines rows (filtered [start,end)):", len(kl))

    # 2) Binance funding
    print("\nDownloading Binance funding...")
    fd = download_binance_funding(client, cfg)
    print("✅ funding rows (bucketed hour, filtered [start,end)):", len(fd))

    # 3) OI snapshot
    print("\nTaking Binance open interest snapshot...")
    oi = snapshot_binance_open_interest(client, cfg)
    print("✅ openInterest snapshot:", oi)

    # 4) Coinbase candles
    print("\nDownloading Coinbase candles...")
    cb = download_coinbase_candles(client, cfg)
    print("✅ coinbase rows (initial):", len(cb))

    # 4b) Coinbase repair (refetch missing hours)
    print("\nQA Coinbase coverage + targeted refetch if needed...")
    cb_fixed, cb_audit = repair_coinbase_if_needed(client, cfg, cb)
    print("✅ coinbase rows (final):", len(cb_fixed))
    print("   missing hours (final):", cb_audit["final_missing_count"])

    # Save repaired coinbase + audit
    out_cb_dir = os.path.join(cfg.out_root, "raw", "benchmarks", "coinbase_exchange", f"candles_{cfg.coinbase_granularity_s}", cfg.product_coinbase)
    cb_files = save_df(cb_fixed, out_cb_dir, "candles_repaired")
    write_json(os.path.join(out_cb_dir, "repair_audit.json"), cb_audit)
    write_json(os.path.join(out_cb_dir, "repair_files.json"), cb_files)

    # 5) Normalize Binance core + QA
    print("\nNormalizing Binance core (hourly) + QA...")
    norm, qa = normalize_binance_core(cfg, kl, fd)
    out_norm_dir = os.path.join(cfg.out_root, "normalized", "cex_core")
    ensure_dir(out_norm_dir)

    norm_files = save_df(norm, out_norm_dir, f"binance_futures_{cfg.symbol_binance.lower()}_{cfg.interval}_normalized")
    write_json(os.path.join(out_norm_dir, "qa_report.json"), qa)

    manifest = {
        "dataset": "normalized_cex_core",
        "config": asdict(cfg),
        "files": norm_files,
        "environment": env_fingerprint(),
        "http_summary": {"requests_made": client.requests_made, "transient_errors": client.errors_transient},
        "notes": "Normalized to expected hourly index [start,end). Funding aligned using bucketed hourly timestamps.",
    }
    write_json(os.path.join(out_norm_dir, "manifest.json"), manifest)

    # Summary
    exp = expected_counts_1h(start_ts, end_ts)
    print("\n================ SUMMARY ================")
    print("Expected klines 1h:", exp["expected_1h_candles"], "| got:", len(kl))
    print("Expected funding ~8h:", exp["expected_funding_8h"], "| got (bucketed):", len(fd))
    print("Coinbase expected 1h:", exp["expected_1h_candles"], "| got final:", len(cb_fixed), "| missing:", cb_audit["final_missing_count"])
    print("Binance normalized missing close hours:", qa["missing_close_hours"])
    print("Binance normalized missing funding_rate hours:", qa["funding_rate_missing_hours"])
    print("Raw root:", cfg.out_root)
    print("Normalized folder:", out_norm_dir)
    print("=========================================\n")


# ---- EXECUTE
run(CFG)